In [16]:
import os
import cv2
import time

import torch
import model.detector
import utils.utils


In [17]:
data_dir = "data/coco.data"
weights_dir = "modelzoo/coco2017-0.241078ap-model.pth"
img_dir = "img/000139.jpg"
cfg = utils.utils.load_datafile(data_dir)
assert os.path.exists(weights_dir), "请指定正确的模型路径"
assert os.path.exists(img_dir), "请指定正确的测试图像路径"

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.detector.Detector(cfg["classes"], cfg["anchor_num"], True).to(device)
model.load_state_dict(torch.load(weights_dir, map_location=device))

#sets the module in eval node
model.eval()

#数据预处理
ori_img = cv2.imread(img_dir)
res_img = cv2.resize(ori_img, (cfg["width"], cfg["height"]), interpolation = cv2.INTER_LINEAR) 
img = res_img.reshape(1, cfg["height"], cfg["width"], 3)
img = torch.from_numpy(img.transpose(0,3, 1, 2))
img = img.to(device).float() / 255.0

#模型推理
start = time.perf_counter()
preds = model(img)
end = time.perf_counter()
time = (end - start) * 1000.
print("forward time:%fms"%time)

#特征图后处理
output = utils.utils.handel_preds(preds, cfg, device)
output_boxes = utils.utils.non_max_suppression(output, conf_thres = 0.3, iou_thres = 0.4)

#加载label names
LABEL_NAMES = []
with open(cfg["names"], 'r') as f:
    for line in f.readlines():
        LABEL_NAMES.append(line.strip())

h, w, _ = ori_img.shape
scale_h, scale_w = h / cfg["height"], w / cfg["width"]

#绘制预测框
for box in output_boxes[0]:
    box = box.tolist()
    
    obj_score = box[4]
    category = LABEL_NAMES[int(box[5])]

    x1, y1 = int(box[0] * scale_w), int(box[1] * scale_h)
    x2, y2 = int(box[2] * scale_w), int(box[3] * scale_h)

    cv2.rectangle(ori_img, (x1, y1), (x2, y2), (255, 255, 0), 2)
    cv2.putText(ori_img, '%.2f' % obj_score, (x1, y1 - 5), 0, 0.7, (0, 255, 0), 2)	
    cv2.putText(ori_img, category, (x1, y1 - 25), 0, 0.7, (0, 255, 0), 2)

cv2.imwrite("test_result.png", ori_img)


load param...
forward time:14.497800ms


-1

In [19]:
cv2.imshow("test",ori_img)
cv2.waitKey(0)

-1